# Simran Amesar
### Matriculation number - 	100007050

### Task: 
Create a small search or recommendation system that matches a user query with the most relevant text documents using TF-IDF and cosine similarity.

### Create the Dataset

In [1]:
movies = [
    {
        "title": "Interstellar",
        "description": "A team of astronauts travels through a wormhole in space to find a new home for humanity, facing black holes and time dilation."
    },
    {
        "title": "WALL-E",
        "description": "A lonely robot on a deserted Earth collects trash until a sleek robot arrives from space, sparking an adventure across the galaxy."
    },
    {
        "title": "The Avengers",
        "description": "A team of superheroes including Iron Man, Thor, and Captain America unite to battle an alien invasion threatening Earth."
    },
    {
        "title": "Julie & Julia",
        "description": "A young blogger cooks every recipe from Julia Child's cookbook, discovering her passion for French cuisine and healthy gourmet food."
    },
    {
        "title": "Chef",
        "description": "A talented chef quits his restaurant job and starts a food truck, serving delicious street food on a road trip across America."
    },
    {
        "title": "The Martian",
        "description": "An astronaut is stranded alone on Mars and must use science and ingenuity to survive until a rescue mission can reach him."
    },
    {
        "title": "Ratatouille",
        "description": "A talented rat in Paris dreams of becoming a chef and secretly controls a young kitchen worker to cook exquisite gourmet meals."
    },
    {
        "title": "Star Wars: A New Hope",
        "description": "A young farm boy joins rebel fighters and a wise Jedi knight to battle an evil empire and destroy a massive space weapon."
    },
    {
        "title": "Terminator 2",
        "description": "A reprogrammed robot from the future is sent back in time to protect a boy from a more advanced and deadly robot assassin."
    },
    {
        "title": "Gravity",
        "description": "Two astronauts struggle to survive in outer space after their shuttle is destroyed by debris, trying desperately to return to Earth."
    }
]

print(f"Dataset created: {len(movies)} movies loaded.")
for i, m in enumerate(movies, 1):
    print(f"  {i:>2}. {m['title']}")

Dataset created: 10 movies loaded.
   1. Interstellar
   2. WALL-E
   3. The Avengers
   4. Julie & Julia
   5. Chef
   6. The Martian
   7. Ratatouille
   8. Star Wars: A New Hope
   9. Terminator 2
  10. Gravity


In [3]:
import re
import nltk
import numpy as np
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

### Text Preprocessing

In [4]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # 1. Lowercase
    text = text.lower()

    # 2. Remove punctuation (keep only letters and spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # 3. Tokenize and remove stopwords
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]

    # 4. Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return ' '.join(tokens)

processed_docs = [preprocess(m['description']) for m in movies]

print("Preprocessing preview:\n")
for i in range(3):
    print(f"[{movies[i]['title']}]")
    print(f"  ORIGINAL : {movies[i]['description']}")
    print(f"  PROCESSED: {processed_docs[i]}")
    print()

Preprocessing preview:

[Interstellar]
  ORIGINAL : A team of astronauts travels through a wormhole in space to find a new home for humanity, facing black holes and time dilation.
  PROCESSED: team astronaut travel wormhole space find new home humanity facing black hole time dilation

[WALL-E]
  ORIGINAL : A lonely robot on a deserted Earth collects trash until a sleek robot arrives from space, sparking an adventure across the galaxy.
  PROCESSED: lonely robot deserted earth collect trash sleek robot arrives space sparking adventure across galaxy

[The Avengers]
  ORIGINAL : A team of superheroes including Iron Man, Thor, and Captain America unite to battle an alien invasion threatening Earth.
  PROCESSED: team superheroes including iron man thor captain america unite battle alien invasion threatening earth



### Build TF-IDF Matrix

In [5]:
vectorizer = TfidfVectorizer()

# Fit on the corpus and transform all documents
tfidf_matrix = vectorizer.fit_transform(processed_docs)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]} documents × {tfidf_matrix.shape[1]} unique terms")
print(f"\nSample vocabulary (first 15 terms):")
print(vectorizer.get_feature_names_out()[:15])

TF-IDF matrix shape: (10, 116)
  → 10 documents × 116 unique terms

Sample vocabulary (first 15 terms):
['across' 'advanced' 'adventure' 'alien' 'alone' 'america' 'arrives'
 'assassin' 'astronaut' 'back' 'battle' 'becoming' 'black' 'blogger' 'boy']


### Search Function

In [6]:
def search(query, top_n=3):
    processed_query = preprocess(query)
    print(f"Query        : '{query}'")
    print(f"Preprocessed : '{processed_query}'")
    print("-" * 55)
    query_vector = vectorizer.transform([processed_query])

    # Compute cosine similarity between query and all documents
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # Sort by score descending, take top N
    ranked_indices = np.argsort(scores)[::-1][:top_n]

    # Build results
    results = []
    for rank, idx in enumerate(ranked_indices, 1):
        results.append({
            "rank"        : rank,
            "title"       : movies[idx]['title'],
            "description" : movies[idx]['description'],
            "score"       : round(float(scores[idx]), 4)
        })
    return results


def display_results(results):
    if all(r['score'] == 0 for r in results):
        print("No relevant results found. Try different keywords.")
        return
    for r in results:
        print(f"#{r['rank']} — {r['title']}")
        print(f"   Score      : {r['score']}")
        print(f"   Description: {r['description']}")
        print()

In [10]:
results = search("space adventure with robots")
display_results(results)

Query        : 'space adventure with robots'
Preprocessed : 'space adventure robot'
-------------------------------------------------------
#1 — WALL-E
   Score      : 0.5318
   Description: A lonely robot on a deserted Earth collects trash until a sleek robot arrives from space, sparking an adventure across the galaxy.

#2 — Terminator 2
   Score      : 0.28
   Description: A reprogrammed robot from the future is sent back in time to protect a boy from a more advanced and deadly robot assassin.

#3 — Gravity
   Score      : 0.0886
   Description: Two astronauts struggle to survive in outer space after their shuttle is destroyed by debris, trying desperately to return to Earth.



### Run Searches

In [9]:
results = search("healthy food")
display_results(results)

Query        : 'healthy food'
Preprocessed : 'healthy food'
-------------------------------------------------------
#1 — Julie & Julia
   Score      : 0.3543
   Description: A young blogger cooks every recipe from Julia Child's cookbook, discovering her passion for French cuisine and healthy gourmet food.

#2 — Chef
   Score      : 0.2772
   Description: A talented chef quits his restaurant job and starts a food truck, serving delicious street food on a road trip across America.

#3 — Gravity
   Score      : 0.0
   Description: Two astronauts struggle to survive in outer space after their shuttle is destroyed by debris, trying desperately to return to Earth.

